# Chennai Traffic Congestion — Research Notebook

Offline companion to the live dashboard. Uses the exact same `backend.simulator` and `backend.clustering` modules as the running app, so every experiment here is reproducible and consistent with what the dashboard shows — no server needs to be running.

Sections:
1. Generate a synthetic live snapshot
2. Congestion distribution
3. k-distance plot to choose DBSCAN `eps`
4. Hotspot clustering (DBSCAN, congestion-weighted, haversine)
5. eps / min_samples sensitivity sweep
6. DBSCAN vs KMeans contrast
7. Full day-cycle congestion simulation per road category

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if (pathlib.Path.cwd() / 'notebooks').exists() is False and pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
if (ROOT / 'backend').exists() is False:
    ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from backend.simulator import TrafficSimulator
from backend.clustering import detect_hotspots, sweep_eps, HotspotParams
from config.chennai_network import ROADS

sns.set_theme(style='darkgrid')
sim = TrafficSimulator(rng_seed=42)

In [ ]:
# 1. Snapshot at a chosen simulated time (evening rush hour on a weekday)
snapshot_time = datetime(2026, 7, 13, 18, 30)  # Monday 6:30pm
pings = sim.tick(snapshot_time)
df = pd.DataFrame(pings)
df.head()

In [ ]:
# 2. Congestion distribution by road category
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=df, x='category', y='congestion', ax=ax)
ax.set_title(f'Congestion by road category @ {snapshot_time:%H:%M}')
plt.show()

In [ ]:
# 3. k-distance plot — a standard empirical way to pick DBSCAN's eps:
# sort each point's distance to its k-th nearest neighbour and look for the 'elbow'.
from sklearn.neighbors import NearestNeighbors

coords_rad = np.radians(df[['lat', 'lon']].values)
k = 6
nn = NearestNeighbors(n_neighbors=k, metric='haversine').fit(coords_rad)
dist, _ = nn.kneighbors(coords_rad)
kth_dist_km = np.sort(dist[:, -1]) * 6371.0

plt.figure(figsize=(8, 4))
plt.plot(kth_dist_km)
plt.ylabel(f'{k}-NN distance (km)')
plt.xlabel('points sorted by distance')
plt.title('k-distance elbow plot for eps selection')
plt.show()

In [ ]:
# 4. Run the same congestion-weighted DBSCAN the dashboard uses
result = detect_hotspots(pings, HotspotParams(eps_km=0.45, min_samples=6))
hotspots_df = pd.DataFrame(result['hotspots'])
hotspots_df

In [ ]:
# Visualize points + clusters
labels = np.array(result['labels'])
fig, ax = plt.subplots(figsize=(7, 7))
sc = ax.scatter(df['lon'], df['lat'], c=labels, cmap='tab20', s=14)
for h in result['hotspots']:
    ax.scatter(h['centroid']['lon'], h['centroid']['lat'], c='red', marker='x', s=120)
ax.set_title('DBSCAN hotspot clusters (red x = centroid, -1/gray = noise)')
ax.set_xlabel('lon'); ax.set_ylabel('lat')
plt.show()

In [ ]:
# 5. eps sensitivity sweep — how cluster count / noise ratio respond to eps
sweep = sweep_eps(pings, eps_values_km=[0.2, 0.3, 0.4, 0.45, 0.5, 0.6, 0.8, 1.0], min_samples=6)
sweep_df = pd.DataFrame(sweep)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(sweep_df['eps_km'], sweep_df['n_clusters'], 'o-', color='#4da3ff', label='n_clusters')
ax2.plot(sweep_df['eps_km'], sweep_df['noise_ratio'], 's--', color='#e74c3c', label='noise_ratio')
ax1.set_xlabel('eps (km)'); ax1.set_ylabel('n_clusters', color='#4da3ff'); ax2.set_ylabel('noise_ratio', color='#e74c3c')
plt.title('DBSCAN eps sensitivity'); plt.show()
sweep_df

In [ ]:
# 6. DBSCAN vs KMeans contrast — KMeans assumes convex, similarly-sized
# clusters and can't express 'noise'; congestion hotspots are neither convex
# nor uniformly present, which is why DBSCAN is the better fit here.
from sklearn.cluster import KMeans

congested = df[df['congestion'] >= 0.35]
km = KMeans(n_clusters=len(result['hotspots']) or 3, n_init=10, random_state=0)
km_labels = km.fit_predict(congested[['lat', 'lon']])

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].scatter(congested['lon'], congested['lat'], c=labels[df['congestion'] >= 0.35], cmap='tab20', s=14)
axes[0].set_title('DBSCAN (density + congestion weighted)')
axes[1].scatter(congested['lon'], congested['lat'], c=km_labels, cmap='tab20', s=14)
axes[1].set_title('KMeans (forced convex partitions, no noise concept)')
plt.show()

In [ ]:
# 7. Full-day congestion cycle per road category (deterministic time-of-day model)
day_start = datetime(2026, 7, 13, 0, 0)
rows = []
sim2 = TrafficSimulator(rng_seed=7)
for minute in range(0, 24 * 60, 15):
    t = day_start + timedelta(minutes=minute)
    snap = sim2.tick(t)
    by_cat = pd.DataFrame(snap).groupby('category')['congestion'].mean()
    for cat, val in by_cat.items():
        rows.append({'time': t, 'category': cat, 'congestion': val})

day_df = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(11, 5))
for cat, grp in day_df.groupby('category'):
    ax.plot(grp['time'], grp['congestion'], label=cat)
ax.legend(); ax.set_title('Simulated 24h congestion profile by road category')
ax.set_ylabel('mean congestion'); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

## Notes for extending this research

- Swap `TrafficSimulator` for a real feed (e.g. TomTom Traffic Flow API) by producing the same `{id, road_id, lat, lon, congestion, speed_kmph, timestamp}` schema — `detect_hotspots` and the dashboard need no changes.
- `HotspotParams.congestion_floor` controls what counts as 'congested enough to cluster'; lowering it surfaces more, smaller hotspots.
- The forecasting model in `backend/forecasting.py` is an intentionally transparent linear-trend blend — a good baseline to benchmark a heavier model (e.g. Prophet, an LSTM) against.